# Adapters - Comprehensive Implementation Guide

This notebook covers:
- **Basic Implementation**: Simple, educational version
- **Advanced Implementation**: Production-ready patterns
- **Real-World Examples**: How companies use this in production
- **Integration**: Using popular libraries

Source: `llm/concepts/adapters.md`

## Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("Libraries loaded successfully")

## 1. Basic Implementation

Simple, educational version to understand core concepts.

In [ ]:
# Basic Adapter Module Implementation
import torch
import torch.nn as nn

class SimpleAdapter(nn.Module):
    """Minimal adapter implementation: down-project -> activate -> up-project"""

    def __init__(self, hidden_size=768, bottleneck_size=64):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck_size)
        self.up = nn.Linear(bottleneck_size, hidden_size)
        self.activation = nn.GELU()

    def forward(self, x):
        down = self.down(x)
        activated = self.activation(down)
        up = self.up(activated)
        return x + up  # Residual connection

# Test
adapter = SimpleAdapter(hidden_size=768, bottleneck_size=64)
x = torch.randn(2, 10, 768)  # batch=2, seq_len=10, hidden=768
output = adapter(x)
print(f"Input: {x.shape}, Output: {output.shape}")
print(f"Params: {sum(p.numel() for p in adapter.parameters()):,}")

## 2. Advanced Implementation

Production-ready patterns with optimization and error handling.

In [ ]:
# Production Adapter with Layer Norm and Initialization
import torch
import torch.nn as nn
import math

class ProductionAdapter(nn.Module):
    """Production-ready adapter with layer norm and proper initialization"""

    def __init__(self, hidden_size=768, bottleneck_size=64, dropout=0.1):
        super().__init__()
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.down = nn.Linear(hidden_size, bottleneck_size)
        self.up = nn.Linear(bottleneck_size, hidden_size)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.scale = 1.0 / math.sqrt(bottleneck_size)

        # Proper initialization
        nn.init.kaiming_uniform_(self.down.weight)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        residual = x
        x = self.layer_norm(x)
        x = self.down(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.up(x) * self.scale
        return residual + x

# Usage with transformer
adapter = ProductionAdapter(hidden_size=768, bottleneck_size=64)
x = torch.randn(2, 10, 768)
output = adapter(x)
print(f"Params: {sum(p.numel() for p in adapter.parameters()):,}")
print(f"Trainable: {sum(p.numel() for p in adapter.parameters() if p.requires_grad):,}")

## Real-World: Huggingface

How this is used in production systems.

In [ ]:
# Real-World: Using HuggingFace PEFT Library
from peft import get_peft_model, LoraConfig, TaskType
from transformers import AutoModelForCausalLM

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("gpt2")

# Configure adapters (actually using LoRA from PEFT for production)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)

# Wrap model with adapters
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

# Training setup
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=1e-4)

# Dummy batch
input_ids = torch.randint(0, 50257, (2, 10))
outputs = model(input_ids, labels=input_ids)
loss = outputs.loss
loss.backward()
optimizer.step()

print(f"Loss: {loss.item():.4f}")

## Real-World: Multitask

How this is used in production systems.

In [ ]:
# Real-World: Multi-Task Adapter Sharing
class MultiTaskAdapterModel(nn.Module):
    """Shared base model with task-specific adapters"""

    def __init__(self, base_model, num_tasks=3, adapter_dim=64):
        super().__init__()
        self.base = base_model

        # One adapter per task
        self.adapters = nn.ModuleDict({
            f"task_{i}": ProductionAdapter(768, adapter_dim)
            for i in range(num_tasks)
        })

        # Task-specific classifiers
        self.classifiers = nn.ModuleDict({
            f"task_{i}": nn.Linear(768, 2)  # Binary classification
            for i in range(num_tasks)
        })

    def forward(self, input_ids, task_id, attention_mask=None):
        # Shared base
        hidden = self.base(input_ids, attention_mask).last_hidden_state

        # Task-specific adapter
        adapted = self.adapters[f"task_{task_id}"](hidden[:, 0])

        # Task-specific classifier
        logits = self.classifiers[f"task_{task_id}"](adapted)
        return logits

# Usage: same base model, different adapters for different tasks
# Total params = base + (num_tasks * adapter_params) << (base + num_tasks * full_params)

## Resources & Next Steps

- **Detailed Explanation**: See `llm/concepts/adapters.md`
- **Interview Questions**: Review Q&A in markdown file
- **Real-World Examples**: See companies section in markdown
- **Experiment**: Modify the code above and run cells

### Concepts to explore next:
- Related concepts (see markdown file)
- Cross-concept combinations
- Integration with other techniques